# Roman Urdu captions — Colab training

**Runtime → Change runtime type → T4 GPU** before running anything.

Then **Runtime → Run all**. Every cell is idempotent and the whole thing is
resumable, so if the session drops you re-open and run all again — it picks up
from the last checkpoint on Drive rather than starting over.

Free Colab disconnects at roughly four hours, so training is stopped by a clock
(`MAX_TRAIN_MINUTES`), not by an epoch. An unfinished run that saved nothing is
worth zero; a partial adapter is worth continuing.

**Never paste a token into a cell.** A `.ipynb` stores its output, so a printed
token is committed with the file. Use the key icon in the left sidebar to add
`HF_TOKEN` as a Colab secret.

In [ ]:
# --- Settings -------------------------------------------------------------
import os
import subprocess
import time

REPO = "https://github.com/nabeeltahirdeveloper/STT-Model.git"
BRANCH = "phase0-baseline-and-spelling-spec"
HF_MODEL = "MubeenAmjad205/roman-urdu-captions"  # private
TRAIN_HOURS = 20  # audio hours to fetch (~0.62 GB each)
MAX_TRAIN_MINUTES = 150  # leaves room for download, eval and upload in 4 h

T0 = time.time()


def sh(cmd, **kw):
    print(f"$ {cmd}", flush=True)
    return subprocess.run(cmd, shell=True, check=kw.pop("check", True), **kw)


def elapsed():
    return f"{(time.time() - T0) / 60:.0f} min elapsed"

In [ ]:
# --- GPU check ------------------------------------------------------------
# Fail here rather than 40 minutes in. A CPU runtime will "work" and take days.
import torch

assert (
    torch.cuda.is_available()
), "No GPU. Runtime -> Change runtime type -> T4 GPU, then Run all again."
name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"{name} · {vram:.0f} GB VRAM")

In [ ]:
# --- Drive: the only thing that survives a disconnect ----------------------
from google.colab import drive

drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/roman-urdu-captions"
ADAPTER = f"{WORK}/adapter"
os.makedirs(ADAPTER, exist_ok=True)
print("checkpoints ->", ADAPTER)

In [ ]:
# --- Repo + dependencies --------------------------------------------------
if not os.path.isdir("/content/model"):
    sh(f"git clone --branch {BRANCH} --single-branch {REPO} /content/model")
os.chdir("/content/model")
sh("git pull --ff-only", check=False)

sh("pip -q install uv")
sh("uv sync --extra train --quiet", check=False)
# bitsandbytes gives an 8-bit optimizer: optimizer state drops 6.2 GB -> 1.6 GB,
# which is what makes full fine-tuning fit on a 16 GB T4 rather than forcing LoRA.
sh("uv pip install -q bitsandbytes accelerate", check=False)
print(elapsed())

In [ ]:
# --- Auth: from Colab secrets, never from a cell --------------------------
from google.colab import userdata

try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    from huggingface_hub import whoami

    print("hugging face:", whoami()["name"])
except Exception as e:
    print("No HF_TOKEN secret — training will run, upload will be skipped.")
    print("Add it with the key icon in the left sidebar.", type(e).__name__)

In [ ]:
# --- Data -----------------------------------------------------------------
# Transcripts are tiny; audio is the slow part. Both download inside Google's
# network, which is far faster than a home connection. Resumable: re-running
# skips whatever is already on disk.
sh(
    "uv run --with huggingface_hub hf download ASLP-lab/UrduSpeech "
    '--repo-type dataset --include "corpus/**/*_final_transcription.jsonl" '
    "--local-dir data/raw/urduspeech",
    check=False,
)

sh(f"uv run python -m scripts.select_training_subset --hours {TRAIN_HOURS}")
sh(
    "uv run python -m scripts.fetch_training_audio --manifest data/labels/train-subset.jsonl",
    check=False,
)
print(elapsed())

In [ ]:
# --- Train ----------------------------------------------------------------
# Stops on the clock and saves. If the session dropped earlier, --resume picks
# up the adapter from Drive instead of starting from the base weights.
resume = f"--resume {ADAPTER}" if os.path.exists(f"{ADAPTER}/adapter_config.json") else ""
remaining = max(10, MAX_TRAIN_MINUTES - (time.time() - T0) / 60)
sh(
    f"uv run python -m scripts.train --out {ADAPTER} {resume} "
    f"--max-minutes {remaining:.0f} --save-every 100"
)
print(elapsed())

In [ ]:
# --- The gate: CER, not script mix ----------------------------------------
# This session's hard lesson. Script mix went 5.7% -> 100% Latin across three
# runs and was reported as success; CER had meanwhile gone 34.9% -> 64.1%, i.e.
# the model got twice as wrong while looking twice as good. Accuracy is the
# number. See experiments/lora_mps/README.md.
sh(
    "uv run --with huggingface_hub hf download ASLP-lab/UrduSpeech "
    '--repo-type dataset --include "benchmark/**" '
    "--local-dir data/raw/urduspeech",
    check=False,
)
sh(
    f"uv run python -m experiments.lora_mps.evaluate_adapter --adapter {ADAPTER} --clips 12",
    check=False,
)

In [ ]:
# --- Upload ---------------------------------------------------------------
# Drive is one copy, not a backup. The private HF repo is the second.
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import HfApi, create_repo

    create_repo(HF_MODEL, private=True, exist_ok=True)
    HfApi().upload_folder(
        folder_path=ADAPTER,
        repo_id=HF_MODEL,
        commit_message=f"adapter after {elapsed()}",
    )
    print("uploaded ->", HF_MODEL, "(private)")
else:
    print("No HF_TOKEN — adapter is on Drive only, at", ADAPTER)
print(elapsed())

## If the session dies

Re-open this notebook and **Run all**. The repo is already cloned, the audio is
already downloaded, and training resumes from the Drive checkpoint. Nothing is
repeated that does not need to be.

## Reading the result

The only number that matters is **CER against the eval set**. Compare it with
stock 0.6B plus our romanizer, which scored **34.9%** on the same clips.

- **Below 34.9%** — training helped. That is the first real evidence it does.
- **Above** — it did not, whatever the script mix says. Lower the learning rate
  (`--learning-rate 2e-5`) and check the loss curve is falling rather than
  rising before spending another session on it.

Do not compare against the 27.9% baseline: that is the 1.7B model on the full
set, and ADR-016 explains why the comparison is not like for like.